# Day 7 · 作品集整合 · 德邦网点经营分析报告 · 海豚生 🐬

> **7 天练习收官** — 把 Day 1~6 合成一份完整经营分析报告  
> 可直接放 GitHub 作品集，也可用于模拟周会/月会汇报

---

## 报告结构

| 章节 | 来源 | 回答的问题 |
|------|------|-------------|
| 1. KPI 总览 | Day 1 | 整体经营怎么样？ |
| 2. 区域诊断 | Day 2 | 哪个区有问题？ |
| 3. 客户 ABC | Day 3 | 谁是大客户？ |
| 4. 产品 & 趋势 | Day 4+6 | 推什么产品？趋势如何？ |
| 5. 服务质量 | Day 5 | 准时吗？客户满意吗？ |
| 6. 综合诊断 | 全部 | 亮点 / 风险 / 行动建议 |

**作者：** 德邦经营方向 · 海豚生  
**数据：** `data/orders.csv`（150 条德邦向示例运单）

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei"]
plt.rcParams["axes.unicode_minus"] = False

# 基准线（和 toolkit 一致）
BENCHMARK = {"margin": 25.0, "on_time": 75.0, "sat": 4.0, "complaint": 3.0, "rev_per_kg": 1.5}
SLA_DAYS = 3

df = pd.read_csv("data/orders.csv")
df["毛利"] = df["revenue"] - df["cost"]
df["month"] = pd.to_datetime(df["date"]).dt.to_period("M").astype(str)
df["准时"] = df["delivery_days"] <= SLA_DAYS
total_revenue = df["revenue"].sum()

print("数据加载完成 ✅")
print(f"  运单: {len(df)} 条 | 客户: {df['customer_name'].nunique()} 个 | 区域: {df['region'].nunique()} 个")
print(f"  时间: {df['month'].min()} ~ {df['month'].max()}")

---
## 第 1 章 · KPI 总览（Day 1）

In [ ]:
billing_weight = df["billing_weight_kg"].fillna(df["weight_kg"]).sum()
kpi = {
    "总营收(元)": round(df["revenue"].sum(), 2),
    "总单量": len(df),
    "毛利(元)": round(df["毛利"].sum(), 2),
    "毛利率(%)": round(df["毛利"].sum() / df["revenue"].sum() * 100, 2),
    "单公斤营收(元)": round(df["revenue"].sum() / billing_weight, 3),
    "准时率(%)": round(df["准时"].mean() * 100, 1),
    "平均满意度": round(df["satisfaction"].mean(), 2),
    "客诉率(%)": round(df["is_complaint"].mean() * 100, 1),
}

print("=" * 45)
print("  核心 KPI · 海豚生")
print("=" * 45)
for k, v in kpi.items():
    print(f"  {k:12s}: {v}")
print("=" * 45)
pd.Series(kpi).to_frame("数值")

---
## 第 2 章 · 区域诊断（Day 2）

In [ ]:
region = df.groupby("region").agg(总营收=("revenue","sum"), 总单量=("order_id","count"), 总毛利=("毛利","sum")).reset_index()
region["毛利率(%)"] = (region["总毛利"] / region["总营收"] * 100).round(2)
region = region.sort_values("总营收", ascending=False)

top_region = region.iloc[0]
worst_margin_region = region.sort_values("毛利率(%)").iloc[0]

print(f"营收冠军: {top_region['region']}区 ({top_region['总营收']/10000:.1f}万)")
print(f"毛利率最低: {worst_margin_region['region']}区 ({worst_margin_region['毛利率(%)']}%)")
region

---
## 第 3 章 · 客户 ABC（Day 3）

In [ ]:
cust = df.groupby("customer_name").agg(总营收=("revenue","sum")).reset_index().sort_values("总营收", ascending=False)
cust["累计占比(%)"] = (cust["总营收"].cumsum() / cust["总营收"].sum() * 100).round(1)
cust["ABC"] = cust["累计占比(%)"].apply(lambda p: "A" if p <= 80 else ("B" if p <= 95 else "C"))

abc_summary = cust.groupby("ABC").agg(客户数=("customer_name","count"), 营收=("总营收","sum"))
abc_summary["占比(%)"] = (abc_summary["营收"] / total_revenue * 100).round(1)

print("TOP 3 客户:", "、".join(cust.head(3)["customer_name"].tolist()))
print(f"A 类 {abc_summary.loc['A','客户数']} 个客户，占营收 {abc_summary.loc['A','占比(%)']}%")
abc_summary

---
## 第 4 章 · 产品 & 趋势（Day 4 + Day 6）

In [ ]:
# 产品毛利率
product = df.groupby("product_type").agg(总营收=("revenue","sum"), 总毛利=("毛利","sum")).reset_index()
product["毛利率(%)"] = (product["总毛利"] / product["总营收"] * 100).round(2)
best_product = product.sort_values("毛利率(%)", ascending=False).iloc[0]

# 月度环比
monthly = df.groupby("month").agg(总营收=("revenue","sum"), 总单量=("order_id","count")).reset_index().sort_values("month")
monthly["环比(%)"] = (monthly["总营收"].pct_change() * 100).round(2)
latest = monthly.iloc[-1]
mom = latest["环比(%)"]

print(f"最高毛利产品: {best_product['product_type']} ({best_product['毛利率(%)']}%)")
print(f"最新月 {latest['month']}: 营收 {latest['总营收']:,.0f} 元，环比 {mom:+.1f}%")
if pd.notna(mom) and mom < -10:
    print("  🔴 触发紧急预警！")

---
## 第 5 章 · 服务质量（Day 5）

In [ ]:
region_svc = df.groupby("region").agg(准时率=("准时","mean"), 客诉率=("is_complaint","mean")).reset_index()
region_svc["准时率(%)"] = (region_svc["准时率"] * 100).round(1)
region_svc["客诉率(%)"] = (region_svc["客诉率"] * 100).round(1)
worst_ontime = region_svc.sort_values("准时率(%)").iloc[0]

print(f"整体准时率: {kpi['准时率(%)']}% (基准 ≥{BENCHMARK['on_time']}%)")
print(f"整体客诉率: {kpi['客诉率(%)']}% (基准 ≤{BENCHMARK['complaint']}%)")
print(f"准时率最低: {worst_ontime['region']}区 ({worst_ontime['准时率(%)']}%)")
region_svc.sort_values("准时率(%)")

---
## 第 6 章 · 综合诊断 Dashboard 📊

一张图看全局（Day 4 技能）

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# 区域营收
axes[0,0].bar(region["region"], region["总营收"], color="#0066CC")
axes[0,0].set_title("区域营收")
axes[0,0].tick_params(axis="x", rotation=30)

# 产品毛利率
p_sorted = product.sort_values("毛利率(%)")
colors = ["#00AA66" if m >= BENCHMARK["margin"] else "#FF9900" for m in p_sorted["毛利率(%)"]]
axes[0,1].barh(p_sorted["product_type"], p_sorted["毛利率(%)"], color=colors)
axes[0,1].axvline(x=BENCHMARK["margin"], color="red", linestyle="--", alpha=0.7)
axes[0,1].set_title("产品毛利率")

# 月度趋势
axes[1,0].plot(monthly["month"], monthly["总营收"], marker="o", color="#0066CC")
axes[1,0].set_title("月度营收趋势")
axes[1,0].tick_params(axis="x", rotation=45)

# ABC 客户
abc_pie = cust.groupby(
    cust["累计占比(%)"].apply(lambda p: "A" if p <= 80 else ("B" if p <= 95 else "C"))
)["总营收"].sum()
axes[1,1].pie(abc_pie, labels=[f"{k}类" for k in abc_pie.index], autopct="%1.1f%%", colors=["#0066CC","#FF9900","#CCC"])
axes[1,1].set_title("ABC 客户营收占比")

plt.suptitle("德邦网点经营分析 Dashboard · 海豚生", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 第 7 章 · 经营诊断报告（最终产出 ⭐）

> 格式和 `logistics-ops-toolkit` 诊断引擎一致：亮点 / 风险 / 行动建议

In [ ]:
highlights, risks, actions = [], [], []

# 自动识别亮点
if kpi["毛利率(%)"] >= BENCHMARK["margin"]:
    highlights.append(f"整体毛利率 {kpi['毛利率(%)']}%，高于基准 {BENCHMARK['margin']}%，盈利模型健康")
highlights.append(f"{top_region['region']}区营收最高（{top_region['总营收']/10000:.1f}万），是网点基本盘")
highlights.append(f"A 类客户 {abc_summary.loc['A','客户数']} 个占营收 {abc_summary.loc['A','占比(%)']}%，{cust.iloc[0]['customer_name']} 等需重点维护")

# 自动识别风险
if kpi["准时率(%)"] < BENCHMARK["on_time"]:
    risks.append(f"准时率 {kpi['准时率(%)']}% 低于基准 {BENCHMARK['on_time']}%，影响客户复购")
if kpi["客诉率(%)"] > BENCHMARK["complaint"]:
    risks.append(f"客诉率 {kpi['客诉率(%)']}% 远高于基准 {BENCHMARK['complaint']}%")
if pd.notna(mom) and mom < -10:
    risks.append(f"最新月 [{latest['month']}] 营收环比下降 {abs(mom)}%，需紧急排查")
risks.append(f"{worst_ontime['region']}区准时率仅 {worst_ontime['准时率(%)']}%，服务质量短板")

# 行动建议
actions = [
    ("紧急" if pd.notna(mom) and mom < -10 else "高", "召开区域经营复盘，分解单量下降原因"),
    ("高", f"对 A 类客户制定续签计划，防止 TOP 客户流失"),
    ("高", f"梳理 {worst_ontime['region']}区超时线路，复制华中区准时经验"),
    ("中", f"加大 {best_product['product_type']} 等高毛利产品销售激励"),
]

# 综合评分
score = 70 + min(len(highlights)*3, 15) - min(len(risks)*4, 30)
score = max(0, min(100, score))

print("=" * 55)
print(f"  德邦网点经营诊断报告 · 海豚生 · 评分 {score}/100")
print("=" * 55)
print("\n【亮点】")
for h in highlights: print(f"  + {h}")
print("\n【风险】")
for r in risks: print(f"  - {r}")
print("\n【行动建议 — 优先执行】")
for p, a in actions: print(f"  [{p}] {a}")
print("=" * 55)

---
## 第 8 章 · 上传 GitHub 作品集 🚀

### 你完成了这些文件

```
data-analysis-master/
├── data/orders.csv                    ← 德邦向运单数据
├── 练习01_Day1_KPI入门_海豚生.ipynb    ← Day 1
├── Day2_Region_Analysis.ipynb         ← Day 2
├── Day3_Customer_ABC.ipynb            ← Day 3
├── Day4_Visualization.ipynb           ← Day 4
├── Day5_Service_Quality.ipynb         ← Day 5
├── Day6_Trend_Alert.ipynb             ← Day 6
└── Day7_Final_Report.ipynb            ← Day 7 ★ 收官
```

### GitHub 上传步骤

1. 打开 [github.com/njq0204/data-analysis](https://github.com/njq0204/data-analysis)
2. **Upload files** 或本地 Git push
3. commit message 建议：
   ```
   feat: 添加德邦经营方向7天实战练习 · 海豚生
   ```
4. 在 README 里加一节「德邦经营实战案例」

### README 推荐文案（复制到 GitHub）

```markdown
## 德邦经营方向实战 · 海豚生 🐬

7 天数据分析练习，基于德邦零担经营场景：

| Day | 主题 | 文件 |
|-----|------|------|
| 1 | KPI 总览 | 练习01_Day1_KPI入门_海豚生.ipynb |
| 2 | 区域诊断 | Day2_Region_Analysis.ipynb |
| 3 | 客户 ABC | Day3_Customer_ABC.ipynb |
| 4 | 可视化汇报 | Day4_Visualization.ipynb |
| 5 | 服务质量 | Day5_Service_Quality.ipynb |
| 6 | 趋势预警 | Day6_Trend_Alert.ipynb |
| 7 | 综合报告 | Day7_Final_Report.ipynb |
```

---

## ✅ 7 天练习全部完成！

恭喜你 🎉 从零散 Notebook 到完整经营分析报告，你已经具备：

- ✅ Pandas 数据处理
- ✅ 德邦经营 KPI 分析框架
- ✅ 区域 / 客户 / 产品三维诊断
- ✅ 可视化汇报能力
- ✅ 趋势预警 + 行动建议
- ✅ GitHub 作品集

**下一步：** 把这些技能用到 `logistics-ops-toolkit` 项目，或换成真实网点数据练手！